In [99]:
import torch
from torchvision import datasets
from torchvision.transforms import v2 as T
from torch.utils.data import DataLoader
import torch.nn as nn
import torchmetrics

In [100]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [101]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

In [102]:
# !wget https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz

In [103]:
train_and_valid_data = datasets.CIFAR10(
    root='data',
    train=True,
        download=True,
    transform=toTensor)
test_data = datasets.CIFAR10(
    root='data',
     train=False,
      download=True,
      transform=toTensor)

In [104]:
train_and_valid_data

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

In [105]:
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data,
    [45_000, 5_000],
)

In [106]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [107]:
X_sample, y_sample = next(iter(train_loader))
X_sample.shape

torch.Size([64, 3, 32, 32])

In [108]:
y_sample.shape

torch.Size([64])

In [109]:
#create a DNN with 20 hidden layers of 100 nurons each

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*32*32, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 10)
)

In [114]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

model.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=100, bias=True)
  (2): SiLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): SiLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): SiLU()
  (7): Linear(in_features=100, out_features=100, bias=True)
  (8): SiLU()
  (9): Linear(in_features=100, out_features=100, bias=True)
  (10): SiLU()
  (11): Linear(in_features=100, out_features=100, bias=True)
  (12): SiLU()
  (13): Linear(in_features=100, out_features=100, bias=True)
  (14): SiLU()
  (15): Linear(in_features=100, out_features=100, bias=True)
  (16): SiLU()
  (17): Linear(in_features=100, out_features=100, bias=True)
  (18): SiLU()
  (19): Linear(in_features=100, out_features=100, bias=True)
  (20): SiLU()
  (21): Linear(in_features=100, out_features=100, bias=True)
  (22): SiLU()
  (23): Linear(in_features=100, out_features=100, bias=True)
  (24): SiLU()
  (25): Linear(in_features=100, out_features

In [111]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [112]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    
    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }
    
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history['train_losses'].append(mean_loss)
        history['train_metrics'].append(metric.compute().item())
        history['valid_metrics'].append(
            evaluate_tm(model, valid_loader, metric).item()
        )

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [113]:
torch.manual_seed(42)
model.to(device)
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(
    task="multiclass",
    num_classes = 10,
).to(device)


_ =train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, 10)

Epoch 1/10, train loss: 2.0453, train metric: 0.2200, valid metric: 0.2804
Epoch 2/10, train loss: 1.8660, train metric: 0.2983, valid metric: 0.2776
Epoch 3/10, train loss: 1.7941, train metric: 0.3373, valid metric: 0.3498
Epoch 4/10, train loss: 1.7287, train metric: 0.3684, valid metric: 0.3696
Epoch 5/10, train loss: 1.6807, train metric: 0.3900, valid metric: 0.3746
Epoch 6/10, train loss: 1.6424, train metric: 0.4065, valid metric: 0.3846
Epoch 7/10, train loss: 1.6112, train metric: 0.4206, valid metric: 0.4204
Epoch 8/10, train loss: 1.5816, train metric: 0.4294, valid metric: 0.4274
Epoch 9/10, train loss: 1.5591, train metric: 0.4398, valid metric: 0.4382
Epoch 10/10, train loss: 1.5347, train metric: 0.4511, valid metric: 0.4222


In [118]:
#first model with batchNorm
model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*32*32, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 100, bias=False), nn.BatchNorm1d(100), nn.SiLU(),
    nn.Linear(100, 10),
)

model2.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=100, bias=False)
  (2): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): SiLU()
  (4): Linear(in_features=100, out_features=100, bias=False)
  (5): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): SiLU()
  (7): Linear(in_features=100, out_features=100, bias=False)
  (8): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (9): SiLU()
  (10): Linear(in_features=100, out_features=100, bias=False)
  (11): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (12): SiLU()
  (13): Linear(in_features=100, out_features=100, bias=False)
  (14): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (15): SiLU()
  (16): Linear(in_features=100, out_features=100, bias=False)
  (17): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, t

In [119]:
torch.manual_seed(42)
model2.to(device)
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(
    task="multiclass",
    num_classes = 10,
).to(device)


_ =train(model2, optimizer, xentropy, accuracy, train_loader, valid_loader, 10)

Epoch 1/10, train loss: 1.9850, train metric: 0.2716, valid metric: 0.3522
Epoch 2/10, train loss: 1.7267, train metric: 0.3766, valid metric: 0.4112
Epoch 3/10, train loss: 1.6133, train metric: 0.4240, valid metric: 0.4056
Epoch 4/10, train loss: 1.5354, train metric: 0.4512, valid metric: 0.4274
Epoch 5/10, train loss: 1.4839, train metric: 0.4694, valid metric: 0.4098
Epoch 6/10, train loss: 1.4480, train metric: 0.4855, valid metric: 0.4594
Epoch 7/10, train loss: 1.3945, train metric: 0.5046, valid metric: 0.4394
Epoch 8/10, train loss: 1.3577, train metric: 0.5197, valid metric: 0.4644
Epoch 9/10, train loss: 1.3238, train metric: 0.5340, valid metric: 0.4508
Epoch 10/10, train loss: 1.2891, train metric: 0.5453, valid metric: 0.4730
